In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import colorir as cl
import numpy as np
from analyses import io

In [ ]:
colors = cl.StackPalette.load("safe")

In [ ]:
celldf = io.read_celldfs(
    "../runs/invading_concat", 
    levels=["replica", "cell_energy", "mul_energy"],
    low_memory=True,
    scan=True,
    concated=True
).with_columns(
    pl.col("cell_energy").str.strip_prefix("cell_energy-").cast(pl.UInt32),
    pl.col("mul_energy").str.strip_prefix("mul_energy-").cast(pl.UInt32),
    pl.col("replica").cast(pl.UInt32),
    displ=(pl.col("center_x") ** 2 + pl.col("center_y") ** 2) ** 0.5,
    chem=pl.col("chem_mass") / pl.col("area")
).with_columns(
    mul_gamma=20 - pl.col("mul_energy"),
    cell_gamma=20 - pl.col("cell_energy"),
).collect()
celldf

In [ ]:
grouppers = ["mul_gamma", "cell_gamma"]
clusterdf = celldf.filter(pl.col("wtime") >= 4e6).group_by(grouppers + ["lineage"]).agg(
    cluster_x=pl.col("center_x").mean(),
    cluster_y=pl.col("center_y").mean(),
    cluster_displ=pl.col("displ").mean(),
    cluster_chem=pl.col("chem").mean()
).sort(grouppers)
clusterdf

In [ ]:
pvdf = clusterdf.filter(
    # pl.col("mul_gamma") < 12
).pivot(
    on="mul_gamma", 
    index=["cell_gamma", "lineage"], 
    values="cluster_chem"
).sort("cell_gamma", "lineage")
pvdf

In [ ]:
ndf = celldf.group_by(grouppers).agg(
    r=pl.col("replica").n_unique()
)
px.scatter(
    ndf,
    x="cell_gamma",
    y="mul_gamma",
    color="r"
)

In [ ]:
x = pvdf["cell_gamma"].unique()
y = pvdf.columns[2:]
dropped = pvdf.drop("cell_gamma", "lineage").to_numpy()
ma = dropped[::2].T
pa = dropped[1::2].T
# It could make sense here to use ma/pa instead, 
# representing how much more likely m is to be picked than p (follows from the Binom distr)
# problem is that this leaves the color map unbounded, so we would have to use log(pa/ma) 
# and apply the same normalisation we use now
a = Mount Etna, or simply Etna, is an active stratovolcano on the east coast o # == np.log(ma/pa)
diff = a / np.nanmax(np.abs(a), axis=1).reshape((-1, 1))
maxi = max(abs(diff.max()), abs(diff.min()))
fig = go.Figure(go.Heatmap(
    z=diff,
    colorscale=cl.Grad(cl.StackPalette.load("curl")[1:-1][::-1]).to_plotly_colorscale(),
    zmax=1,
    zmin=-1
)).update_layout(
    template="plotly_white",
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    xaxis_constrain="domain",
    xaxis_tickvals=np.arange(len(x)),
    xaxis_ticktext=x,
    yaxis_tickvals=np.arange(len(y)),
    yaxis_ticktext=y,
    xaxis_title="cell_gamma",
    yaxis_title="mul_gamma",
    # width=400,
    height=300
)
io.save_plot(fig, "../plots/lineage_dist_to_peak_diff")

In [ ]:
filterdf = celldf.filter(
    pl.col("wtime") >= 4e6,
    mul_energy=12
)
fig = px.violin(
    filterdf.filter(lineage="mul"),
    x="cell_gamma",
    y="displ",
    color="lineage",
    color_discrete_sequence=colors
).update_traces(
    jitter=1,
    marker_line_width=1,
    marker_line_color="white",
    marker_opacity=0.5
).add_traces(
    px.scatter(
        filterdf.filter(lineage="mul").group_by(
            "cell_gamma", 
            "lineage",
            maintain_order=True
        ).median().with_columns(
            x=pl.col("cell_gamma")# .cast(pl.Int32) + pl.when(pl.col("lineage") == "mul").then(-0.18).otherwise(0.18)
        ),
        x="x",
        y="displ",
        color="lineage",
        color_discrete_sequence=colors
    ).data
).update_layout(
    template="plotly_white",
    width=450,
    height=300,
    showlegend=False,
    xaxis_dtick=1,
    # yaxis_range=[0, max_chem],
    yaxis_title="distance to peak"
)
# io.save_plot(fig, "../plots/steady-mixed-pop")
fig

In [ ]:
diffdf = filterdf.group_by(
    "cell_gamma", 
    "lineage",
    maintain_order=True
).mean().sort("cell_gamma")
diff = diffdf.filter(lineage="uni")["displ"] - diffdf.filter(lineage="mul")["displ"]
px.line(
    y=diff,
    x=diffdf["cell_gamma"].unique()
).update_layout(
    template="plotly_white",
    width=400,
    height=300,
    xaxis_title="cell_gamma",
    yaxis_title="mean_p - mean_m",
    xaxis_dtick=1
)

In [ ]:
from statsmodels.stats.weightstats import ttest_ind
filterdf = celldf.filter(
    pl.col("wtime") >= 4e6,
    mul_energy=12,
    cell_gamma=7,
)
_, p, _ = ttest_ind(
    filterdf.filter(
        lineage="mul"
    )["displ"], 
    filterdf.filter(
        lineage="uni"
    )["displ"]
)
p

In [ ]:
grouppers = ["mul_gamma", "cell_gamma", "mul_n"]
df = pl.read_csv("../comp_results/report.csv").with_columns(
    mul_gamma=20 - pl.col("mul_energy"),
    cell_gamma=20 - pl.col("cell_energy")
).sort(grouppers)
df

In [ ]:
gdf = df.group_by(grouppers, maintain_order=True).agg(
    pl.col("mul_won").mean(),
    count=pl.col("mul_won").count()
)
gdf

In [ ]:
px.scatter(
    gdf.filter(pl.col("mul_n") == 15),
    x="mul_gamma",
    y="cell_gamma",
    color="mul_won"
)

In [ ]:
# Please check that all the sims have finished for the final version of the figure
g8df = df.filter(mul_n=15, mul_gamma=8).drop_nulls()
g8df

In [ ]:
fig = px.histogram(
    g8df,
    x="cell_gamma",
    histfunc="count",
    color="mul_won",
    color_discrete_sequence=["#8CB369", "#963242"],
    barmode="stack"
).update_traces(
    marker_line_width=1,
    marker_line_color="white"
).update_layout(
    template="plotly_white",
    bargap=0.05,
    width=350,
    height=250,
    yaxis_dtick=2.5,
    xaxis_dtick=1,
)
io.save_plot(fig, "../plots/competition")
fig

In [ ]:
import statsmodels.api as sm

x = a[2][~np.isnan(a[2])]
y = gg8df["mul_won"][:-1]
X = sm.add_constant(x)
model = sm.OLS(y.to_numpy(), X).fit()
predictions = model.get_prediction(X)
summary = predictions.summary_frame(alpha=0.05)
print(model.summary(alpha=0.05))

In [ ]:
gg8df = gdf.filter(mul_gamma=8, mul_n=15)
px.scatter(
    x=x,
    y=y
).update_traces(
    marker_color=colors[0]
).update_layout(
    template="plotly_white",
    width=300,
    height=250
)